In [1]:
%pip install -q "transformers==4.57.1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import hashlib
import json
import random
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

In [ ]:
MLLM = Path("/content/drive/MyDrive/MLLM/mordern_llm")
D_ROOT = Path("/content/drive/MyDrive/MLLM/section_d")
D_DATA = D_ROOT / "data"
D_DATA.mkdir(parents=True, exist_ok=True)

C_DATA = MLLM / "data/sroie"
manifest = json.loads((C_DATA / "manifest.json").read_text(encoding="utf-8"))
task = json.loads((C_DATA / "task_config.json").read_text(encoding="utf-8"))

TRAIN = manifest["train"]
FIELDS = tuple(task["fields"])
DOC_PROMPT = task["prompt"]
IMAGE_KWARGS = task["image_kwargs"]

processor = AutoProcessor.from_pretrained(MLLM / "outputs/qlora_run1/best_adapter", use_fast=False)
processor.tokenizer.padding_side = "right"

print("Section C folder:", MLLM)
print("Train / validation:", len(TRAIN), len(manifest["val"]))
print("Fields:", FIELDS)

Section C folder: /content/drive/MyDrive/MLLM/mordern_llm
Train / validation: 551 61
Fields: ('company', 'date', 'total')


In [ ]:
LOCK_FILE = D_DATA / "heldout_manifest.json"
LOCAL_TEST_PARQUET = None
N_TEST, SEED = 30, 45

used = manifest["train"] + manifest["val"] + manifest["test"]
used_ids = {r["id"] for r in used}
used_hashes = {r["sha256"] for r in used}

if LOCK_FILE.is_file():
    heldout = json.loads(LOCK_FILE.read_text(encoding="utf-8"))

else:
    source = LOCAL_TEST_PARQUET or hf_hub_download(
        repo_id=manifest["dataset"],
        repo_type="dataset",
        revision=manifest["revision"],
        filename="data/test-00000-of-00001.parquet"
    )
    candidates = pq.read_table(source, columns=["key", "image", "entities"]).to_pylist()
    random.Random(SEED).shuffle(candidates)
    
    selected = []
    seen_ids, seen_hashes = used_ids.copy(), used_hashes.copy()
    
    for row in candidates:
        sample_id = row["key"]
        raw = row["image"]["bytes"]
        reference = {field: row["entities"][field] for field in FIELDS}
        
        if not raw or not all(isinstance(v, str) and v.strip() for v in reference.values()):
            continue
        
        digest = hashlib.sha256(raw).hexdigest()
        if sample_id in seen_ids or digest in seen_hashes:
            continue
        
        rel_path = Path("data/heldout_images") / f"{sample_id}.jpg"
        selected.append((
            {"id": sample_id, "sha256": digest, "reference": reference,
            "image_path": str(rel_path)}, raw
        ))
        seen_ids.add(sample_id)
        seen_hashes.add(digest)

        if len(selected) == N_TEST:
            break

    for record, raw in selected:
        path = D_ROOT / record["image_path"]
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(raw)

    heldout = {
        "dataset": manifest["dataset"],
        "revision": manifest["revision"],
        "seed": SEED,
        "test": [record for record, _ in selected]
    }
    LOCK_FILE.write_text(json.dumps(heldout, ensure_ascii=False, indent=2), encoding="utf-8")

D_TEST = heldout["test"]
print("Locked held-out receipts:", len(D_TEST))
print("Saved:", LOCK_FILE)

Locked held-out receipts: 30
Saved: /content/drive/MyDrive/MLLM/section_d/data/heldout_manifest.json
